# Deep learning on generated credit-card datasets

Upload the prepared local bundle. It contains the original CSV, five generated-data variants plus the baseline on each of five saved splits, and the current code. The training matrix is 60 fits using an MLP classifier and a Transformer classifier.

Choose a GPU runtime in Colab. Training requires setting `RUN_TRAINING = True` in the dedicated cell. The delivered notebook has never been executed. See the bundled `colab/README.md` for recipes, evidence and resume behavior.

In [ ]:
import hashlib
import json
import zipfile
from pathlib import Path

from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload only credit_risk_colab.zip")
archive_path = next(iter(uploaded))
PROJECT = Path("/content/credit_risk_project")
PROJECT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive_path) as archive:
    for member in archive.infolist():
        destination = (PROJECT / member.filename).resolve()
        if PROJECT.resolve() not in destination.parents:
            raise ValueError("Invalid archive path")
    archive.extractall(PROJECT)
manifest = json.loads((PROJECT / "COLAB_BUNDLE_MANIFEST.json").read_text())
for name, expected in manifest.items():
    if hashlib.sha256((PROJECT / name).read_bytes()).hexdigest() != expected:
        raise ValueError(f"Bundle hash mismatch: {name}")
print(f"Verified {len(manifest)} bundled files")

In [ ]:
import os
import subprocess
import sys

os.chdir(PROJECT)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=True)

## Validate saved inputs

This cell checks the original CSV, all saved splits and training tables. It starts no model fitting and writes no experiment results.

In [ ]:
subprocess.run([sys.executable, "-m", "creditrisk.deep_learning", "--validate-only"], check=True)

## Persistent output

Mount Drive and choose the output folder. Use a fresh folder for a new experiment. For an interrupted run, retain the same folder and set `RESUME = True` below. Only completed classifier jobs are reused. An interrupted classifier starts again at epoch one.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT = Path("/content/drive/MyDrive/credit_risk/deep_learning_comparison")

## Explicit training launch

The default switch leaves training disabled. Set it to `True` when ready. The command requests CUDA, trains both neural classifiers across the saved datasets and splits, saves evidence and builds the neural report.

In [ ]:
RUN_TRAINING = False
RESUME = False

command = [
    sys.executable,
    "-m",
    "creditrisk.deep_learning",
    "--input",
    "results/credit_card",
    "--output",
    str(OUTPUT),
    "--device",
    "cuda",
    "--epochs",
    "100",
    "--patience",
    "15",
    "--batch-size",
    "512",
]
if RESUME:
    command.append("--resume")
print("Prepared command", command)
if RUN_TRAINING:
    subprocess.run(command, check=True)
else:
    print("Training disabled. Set RUN_TRAINING = True to launch.")

## Read the completed comparison

The generated report and all prediction evidence stay in the configured Drive folder. Classical results already in the bundle describe the previous four classifiers. Neural results become available after this new training matrix completes.

In [ ]:
from IPython.display import Markdown, display

report = OUTPUT / "REPORT.md"
if report.exists():
    display(Markdown(report.read_text()))
else:
    print("A completed neural report is not available yet.")